# Detailed Solutions: Survival Analysis Lab

Worked answer key for `02_Skeleton_Practice_Lab.ipynb` (and equivalently, the exercises embedded in
`01_Extended_Lab_Survival_Analysis.ipynb`). Each solution includes the code, the output-level result,
and a short explanation of *why* the answer is what it is — not just the mechanical answer.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import datasets, KaplanMeierFitter, ExponentialFitter, WeibullFitter, LogNormalFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index, median_survival_times
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (8, 5)


## Part 1 — Kaplan-Meier

In [ ]:
data = datasets.load_larynx()
data.head()

**Q1.1** — `time` = months survived after diagnosis; `age` = age at diagnosis; `death` = 1 if death was observed, 0 if censored; `Stage_II/III/IV` = 1 if the patient was diagnosed at that stage. If all three stage dummies are 0, the patient was Stage I (the reference/omitted category).

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(data.time, data.death, label='All patients')

fig, ax = plt.subplots(1)
kmf.plot_survival_function(ax=ax)
ci = kmf.confidence_interval_survival_function_
ax.fill_between(ci.index, ci.values[:, 0], ci.values[:, 1], color='gray', alpha=0.3)
ax.set_title('Kaplan-Meier — all patients');

In [ ]:
print('Median survival time:', kmf.median_survival_time_)
print('S(60 months):', kmf.predict(60))

**Q1.2 answer:** The median survival time is the smallest t where S(t) <= 0.5; lifelines reads this directly off the step function. `predict(60)` returns the last known survival estimate at or before t=60 — Kaplan-Meier never interpolates or extrapolates beyond observed jump points.

In [ ]:
split = 65
data['older'] = (data.age >= split).astype(int)
older = data[data.older == 1]
younger = data[data.older == 0]

older_kmf = KaplanMeierFitter().fit(older.time, older.death, label='older')
younger_kmf = KaplanMeierFitter().fit(younger.time, younger.death, label='younger')

fig, ax = plt.subplots(1)
older_kmf.plot(ax=ax); younger_kmf.plot(ax=ax)
ax.set_title('Kaplan-Meier by age group');

In [ ]:
lr_age = logrank_test(older.time, younger.time, older.death, younger.death)
lr_age.print_summary()

**Q1.3 answer:** H0: the two age groups have the same underlying survival distribution (equivalently, hazard ratio = 1). With p typically > 0.05 for this split, we fail to reject H0 — consistent with the book's visual read that age does not separate the curves. This confirms a chart impression with an actual test rather than eyeballing.

In [ ]:
stage_cols = ['Stage_II', 'Stage_III', 'Stage_IV']
data['Stage_I'] = (1 - data[stage_cols].sum(axis=1)).clip(lower=0)
stage_data = {'stage_i': data[data.Stage_I == 1]}
for s in stage_cols:
    stage_data[s.lower()] = data[data[s] == 1]

fig, ax = plt.subplots(1, figsize=(9, 6))
for name in sorted(stage_data):
    d = stage_data[name]
    KaplanMeierFitter().fit(d.time, d.death, label=name).plot(ax=ax, ci_show=False)
ax.set_title('Kaplan-Meier by stage');

In [ ]:
groups, durations, events = [], [], []
for name, d in stage_data.items():
    groups += [name] * len(d)
    durations += list(d.time)
    events += list(d.death)

mv = multivariate_logrank_test(durations, groups, events)
mv.print_summary()

**Q1.4 answer:** The multivariate log-rank test returns a single chi-square statistic testing whether *any* of the four stage groups differs from the others. Given the visibly steep drop for Stage IV in the chart, this test is almost always significant (p < 0.05) — confirming stage matters somewhere, even though the pairwise Stage I vs. Stage II difference alone is small.

## Part 2 — Exponential Model

In [ ]:
exf = ExponentialFitter().fit(data.time, data.death, label='Exponential')

fig, ax = plt.subplots(1)
exf.plot_survival_function(ax=ax)
kmf.plot_survival_function(ax=ax, ci_show=False)
ax.set_title('Exponential vs. Kaplan-Meier');

print('lambda:', exf.lambda_)
print('implied mean survival:', 1/exf.lambda_)

**Q2.1 answer:** lambda is the constant hazard rate; 1/lambda is the mean survival time implied under a *constant*-hazard assumption. Visually, the exponential curve usually sits below the Kaplan-Meier curve in the early period (where the true hazard is lower than average) and above it later (where the true hazard is higher, e.g. after stage-IV patients start dying) — evidence the constant-hazard assumption is only a rough approximation for this cohort.

In [ ]:
candidates = {'Exponential': ExponentialFitter(), 'Weibull': WeibullFitter(), 'LogNormal': LogNormalFitter()}
aic = {}
for name, f in candidates.items():
    f.fit(data.time, data.death, label=name)
    aic[name] = f.AIC_
pd.Series(aic).sort_values()

**Q2.2 answer:** Weibull (which allows the hazard to increase or decrease monotonically over time, rather than staying flat) typically achieves a lower AIC than the plain Exponential model on this data, indicating the constant-hazard assumption is not fully supported — cancer hazard plausibly rises with time since diagnosis for higher stages. This does not mean the exponential model is useless; it means any inference drawn from it (e.g. an extrapolated hazard ratio) should be treated as approximate.

## Part 3 — Cox PH on `larynx`

In [ ]:
cox_data = data[['time', 'death', 'age', 'Stage_II', 'Stage_III', 'Stage_IV']].copy()
cph_larynx = CoxPHFitter()
cph_larynx.fit(cox_data, duration_col='time', event_col='death')
cph_larynx.print_summary()

**Q3.1 answer:** In this dataset, `Stage_IV` (and often `Stage_III`) come out statistically
significant with `exp(coef)` well above 1, while `age` and `Stage_II` are typically not significant.
Interpretation: *"Holding age constant, a Stage IV diagnosis multiplies the hazard of death by
approximately exp(coef) relative to Stage I — a large, significant increase in risk."* Because the
Stage_II hazard ratio's confidence interval usually contains 1, we cannot conclude Stage II differs
meaningfully from Stage I once age is accounted for.

In [ ]:
cph_larynx.plot()
plt.title('Larynx Cox PH — coefficients within 95% CI');

In [ ]:
cph_larynx.check_assumptions(cox_data, p_value_threshold=0.05, show_plots=True)

**Q3.2/3.3 answer:** If a covariate (commonly `Stage_IV`, since its hazard is not just higher but plausibly *front-loaded* — patients who don't die quickly may behave more like other stages) fails the Schoenfeld-residual test, its hazard ratio is not constant over follow-up time, which violates the core Cox PH assumption. Two standard fixes: (1) **stratify** on the offending covariate (`CoxPHFitter().fit(..., strata=['Stage_IV'])`) which lets each stratum have its own baseline hazard while still estimating shared coefficients for the other covariates, or (2) model it as a **time-varying covariate** (interact it with a function of time) using `CoxTimeVaryingFitter`.

## Part 4 — Cox PH on Stanford Heart Transplant Data

In [ ]:
heart = sm.datasets.get_rdataset('heart', package='survival').data
train_data = heart.iloc[0:171]
test_data = heart.iloc[171:]
heart.head()

In [ ]:
kmf_no = KaplanMeierFitter().fit(heart.loc[heart.transplant==0,'stop'], heart.loc[heart.transplant==0,'event'], label='no transplant')
kmf_yes = KaplanMeierFitter().fit(heart.loc[heart.transplant==1,'stop'], heart.loc[heart.transplant==1,'event'], label='transplant')

fig, ax = plt.subplots(1)
kmf_no.plot(ax=ax); kmf_yes.plot(ax=ax)
ax.set_title('Kaplan-Meier Survival Estimates');

print('No-transplant S(300):', kmf_no.predict(300))
print('Transplant S(300):', kmf_yes.predict(300))

**Q4.2 answer:** Transplant recipients show a visibly higher survival probability at day 300 than non-recipients (roughly 0.49 vs. 0.35, matching the book's figures). Based on the raw curves alone it looks like transplantation helps — but see Q5.2 below for why this comparison is confounded.

In [ ]:
lr_results = logrank_test(heart.loc[heart.transplant==0,'stop'], heart.loc[heart.transplant==1,'stop'],
                          heart.loc[heart.transplant==0,'event'], heart.loc[heart.transplant==1,'event'])
lr_results.print_summary()

In [ ]:
cph = CoxPHFitter()
cph.fit(df=train_data[['age','year','surgery','transplant','stop','event']], duration_col='stop', event_col='event')
cph.print_summary()

**Q4.4 answer:** `age`, `year`, and `transplant` are typically significant at the 5% level; `surgery` is borderline (p just above 0.05 even though its 95% CI on the hazard ratio may exclude 1 — a discrepancy the book explicitly calls out, arising because the p-value and the CI are computed from slightly different approximations). *Transplant:* `exp(coef) ~ 0.54` → receiving a transplant is associated with roughly a 46% reduction in hazard, holding age/year/surgery fixed. *Age:* `exp(coef) ~ 1.03` → each additional year of age increases hazard by about 3%.

In [ ]:
cph.plot()
plt.title('Coefficients within 95% Confidence Intervals');

In [ ]:
cph.check_assumptions(train_data[['age','year','surgery','transplant','stop','event']],
                     p_value_threshold=0.05, show_plots=True)

In [ ]:
c_index = concordance_index(train_data['stop'], -cph.predict_partial_hazard(train_data), train_data['event'])
print(f'Concordance index (train): {c_index:.3f}')

**Q4.6 answer:** 0.5 = the model ranks patients no better than a coin flip for who dies/receives-event first; 1.0 = perfect ranking of event order. A concordance around 0.65-0.70 (typical for this model) indicates modest but real discriminative ability — meaningfully better than chance, well short of a clinically decisive predictor.

In [ ]:
cph.predict_survival_function(test_data[['age','year','surgery','transplant']]).plot()
plt.xlabel('Survival Time'); plt.ylabel('Survival Probability'); plt.title('Survival Function for Holdout');

## Part 5 — Synthesis

**Q5.1 answer — comparison table**

| Question | Kaplan-Meier | Exponential | Cox PH |
|---|---|---|---|
| Handles covariates? | No (only stratified subgroups) | No (fit separately per group) | Yes, natively multivariate |
| Assumes a hazard shape? | No — fully non-parametric | Yes — constant hazard | No shape assumed for baseline hazard, but *does* assume proportional hazards across covariate levels |
| Gives a single interpretable effect size? | No (only visual/log-rank group comparison) | Hazard ratio between groups possible, but relies on the constant-hazard assumption | Yes — `exp(coef)` per covariate |
| Assumption to check before trusting it | Sample size / censoring pattern (non-informative censoring) | Constant hazard (check vs. Weibull/AIC, compare to KM) | Proportional hazards (Schoenfeld residuals via `check_assumptions`) |
| Best used for... | Quick, assumption-light description of survival experience; group comparison | Simple parametric summaries / simulation when hazard is plausibly constant | Multivariate risk-factor analysis with interpretable hazard ratios |

**Q5.2 answer:** No — the Cox PH hazard ratio for `transplant` is **not a valid causal estimate as fit
here**. Receiving a transplant requires surviving long enough to receive one (immortal-time /
survivorship bias): patients who die early are systematically classified as "no transplant" even if
they would have received one had they lived longer. This inflates the apparent survival benefit of
transplantation. Correct treatment requires either (a) treating transplant as a **time-varying
covariate** (0 before transplant, 1 after, changing partway through follow-up) or (b) a formal
causal-inference approach (e.g., landmark analysis, inverse-probability-of-treatment weighting). The
book/lab, as written, models transplant as a fixed baseline covariate, which does not correct this bias.

**Q5.3 answer:** With n=90 (larynx) and n=103 (heart), (a) Kaplan-Meier confidence bands are wide,
especially in the tail where few subjects remain at risk — visual group differences that look large can
still be statistically indistinguishable; (b) Cox coefficient standard errors are large relative to a
big dataset, so borderline p-values (like `surgery` above) are genuinely ambiguous rather than a
modeling error; (c) the proportional-hazards check itself is a hypothesis test and therefore also
underpowered at this sample size — *failing* to detect a PH violation with n~100 is weak evidence the
assumption actually holds, and *detecting* one should be taken seriously.
